In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

,user_id,first_name,last_name,email,signup_date,country,referral_source
0,USR_1000,Linda,Smith,linda.smith0@example.com,8/1/23,UK,google_ads
1,USR_1001,David,White,david.white1@example.com,8/12/22,AU,organic
2,USR_1002,Grace,Wilson,grace.wilson2@example.com,3/21/22,US,organic
3,USR_1003,Susan,Davis,susan.davis3@example.com,11/14/24,AU,organic
4,USR_1004,Emma,Miller,emma.miller4@example.com,5/23/24,UK,social_media


In [0]:
@dp.materialized_view(
    name="clean_data_silver_v2",
    comment="Cleaned user data with fixed dates and deduplicated records"
)
def clean_data_silver():
    # Read from bronze table
    df = spark.read.table("data_engineering.etl_process.users_dirty_csv")
    
    # Fix the date column - replace '.' with '/' separator and convert to proper date format
    df = df.withColumn(
        "signup_date", 
        F.to_date(F.regexp_replace(F.col("signup_date"), r"\.", "/"), "MM/dd/yy")
    )
    
    # Remove duplicates based on user_id, keeping the first occurrence
    df_clean = df.dropDuplicates(["user_id"])
    
    return df_clean

In [0]:
# This cell is no longer needed.
# The pipeline automatically manages the table creation and updates.
# To preview data, query the table after running the pipeline.

user_id,first_name,last_name,email,signup_date,country,referral_source
USR_1000,Linda,Smith,linda.smith0@example.com,2023-08-01T00:00:00.000Z,UK,google_ads
USR_1001,David,White,david.white1@example.com,2022-08-12T00:00:00.000Z,AU,organic
USR_1002,Grace,Wilson,grace.wilson2@example.com,2022-03-21T00:00:00.000Z,US,organic
USR_1003,Susan,Davis,susan.davis3@example.com,2024-11-14T00:00:00.000Z,AU,organic
USR_1004,Emma,Miller,emma.miller4@example.com,2024-05-23T00:00:00.000Z,UK,social_media
USR_1005,Grace,Rodriguez,grace.rodriguez5@example.com,2022-01-28T00:00:00.000Z,UK,social_media
USR_1006,Lisa,Rodriguez,lisa.rodriguez6@example.com,2022-11-29T00:00:00.000Z,UK,referral
USR_1007,William,Williams,william.williams7@example.com,2024-03-03T00:00:00.000Z,US,referral
USR_1008,Daniel,Martin,daniel.martin8@example.com,2023-07-10T00:00:00.000Z,US,social_media
USR_1009,Edward,Jones,edward.jones9@example.com,2024-02-29T00:00:00.000Z,US,partner


In [0]:
# This cell is no longer needed.
# The @dp.materialized_view decorator handles writing to the table automatically.
# No need for .write.saveAsTable() operations in Spark Declarative Pipelines.